# AI Garbage Classifier Training Notebook
This notebook is used to train the AI model for garbage classification. It includes data loading, preprocessing, model definition, training, and saving the model.

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt

# Set the dataset path
dataset_path = '../data/garbage_classification'  # Path relative to this notebook's folder

# Load images and labels
images = []
labels = []
class_names = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])

for label in class_names:
    label_path = os.path.join(dataset_path, label)
    if os.path.isdir(label_path):
        for img_file in os.listdir(label_path):
            img_path = os.path.join(label_path, img_file)
            if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                img = cv2.imread(img_path)
                img = cv2.resize(img, (128, 128))  # Resize to 128x128
                img = img / 255.0  # Normalize pixel values
                images.append(img)
                labels.append(class_names.index(label))  # Convert label to integer

# Convert to numpy arrays
X = np.array(images)
y = np.array(labels)

# Split into train/validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Build the model
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(128, 128, 3)),
    layers.MaxPooling2D((2,2)),
    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),
    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(len(class_names), activation='softmax')
])

model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# Train the model
history = model.fit(X_train, y_train, epochs=10, validation_data=(X_val, y_val))

# Save the model
model.save('../models/garbage_model.h5')
print("✅ Model berhasil disimpan di models/garbage_model.h5")

# Plot accuracy and loss
plt.plot(history.history['accuracy'], label='Akurasi Training')
plt.plot(history.history['val_accuracy'], label='Akurasi Validasi')
plt.xlabel('Epoch')
plt.ylabel('Akurasi')
plt.legend()
plt.title('Akurasi Training vs Validasi')
plt.show()